# LLaMA-3.1-8B-Instruct Inference — NL to ASP Translation

## 0 · Login & Imports

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import torch
import json
import os
from pathlib import Path
from transformers import AutoTokenizer, pipeline
from peft import AutoPeftModelForCausalLM
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Libraries loaded.")
print(f"torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 1 · Load Fine-Tuned Adapter

In [ ]:
# ── Update this path to your saved adapter / best checkpoint ──────────────────
adapter_path = "Path/To/Your/Saved/Adapter"

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map="auto",
    torch_dtype=torch.float16,   
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

# LLaMA-3 has no pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded from : {adapter_path}")
# Safe access — PEFT models may not have hf_device_map as a direct attribute
device_map = getattr(model, 'hf_device_map', None) or getattr(model.base_model, 'hf_device_map', 'N/A')
print(f"Device map        : {device_map}")

## 2 · Load Test Dataset

In [ ]:
def load_data(path, test_size=0.1, seed=42):
    """
    Load JSON dataset and convert to conversational format for NL → ASP.
    Expects: {"data_dict": [{"NL_V2": ..., "ASP": ...}, ...]}
    """
    def create_conversation(sample):
        return {
            "messages": [
                {
                    "role": "system",
                    "content": "You are an expert in Translating the Natural language (NL) into Answer Set Programming (ASP) translation. Always provide precise, syntactically and semantically correct translations of NL into ASP."
                },
                {
                    "role": "user",
                    "content": f"Translate the following natural language to answer set programming: {sample['NL_V2']} "
                },
                {
                    "role": "assistant",
                    "content": sample["ASP"]
                },
            ]
        }

    dataset = load_dataset("json", data_files=path, field="data_dict", split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
    print("Dataset converted to conversational format.")

    if test_size == 0:
        return dataset, None

    split_dataset = dataset.train_test_split(test_size=test_size, seed=seed)
    train_dataset = split_dataset["train"].shuffle(seed=seed)
    test_dataset  = split_dataset["test"]
    return train_dataset, test_dataset


In [ ]:
dataset_file = "path/to/your/test_dataset.json"
testset, _ = load_data(dataset_file, test_size=0)

print("Example from test set:")
print(testset[0])
print(f"Test dataset size: {len(testset)}")

## 3 · Decoder Class

In [ ]:
import torch
from transformers import pipeline, PreTrainedTokenizer, AutoModelForCausalLM
from abc import ABC, abstractmethod

os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ─────────────────────────────────────────────────────────────────────────────
# Abstract base
# ─────────────────────────────────────────────────────────────────────────────
class ASPDecoder(ABC):
    @abstractmethod
    def decode(self, prompt: str, max_new_tokens=512, temperature=0) -> str:
        pass


# ─────────────────────────────────────────────────────────────────────────────
# Naive (unconstrained) decoder — NL → ASP directly
# No strip_think_tokens needed — LLaMA-3 has no thinking mode
# ─────────────────────────────────────────────────────────────────────────────
class NaiveASPDecoder(ASPDecoder):

    def __init__(self, model, tokenizer: PreTrainedTokenizer):
        self.tokenizer = tokenizer
        self.model     = model
        self.device    = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.pipeline  = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map=self.device
        )

    def decode(self, prompt: str, max_new_tokens=512, temperature=0) -> str:
        outputs = self.pipeline(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=None,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        generated_text = outputs[0]['generated_text']
        prediction     = generated_text[len(prompt):].strip()

        # Remove EOS token if present at end
        if self.tokenizer.eos_token and self.tokenizer.eos_token in prediction:
            prediction = prediction.split(self.tokenizer.eos_token)[0].strip()

        # LLaMA-3: no think tokens — no strip_think_tokens needed
        return prediction


print("Decoder class defined.")

## 4 · ASPTester (with checkpoint saving)

In [ ]:

class ASPTester:
    def __init__(self, decoder: ASPDecoder, tokenizer: PreTrainedTokenizer, test_dataset,
                 checkpoint_path="eval_checkpoint_llama3.json"):
        self.test_dataset    = test_dataset
        self.decoder         = decoder
        self.tokenizer       = tokenizer
        self.checkpoint_path = checkpoint_path

        if os.path.exists(self.checkpoint_path):
            with open(self.checkpoint_path, 'r') as f:
                checkpoint = json.load(f)
                self.error_count   = checkpoint.get("error_count", 0)
                self.wrong_indexes = checkpoint.get("wrong_indexes", [])
                self.predictions   = checkpoint.get("predictions", [])
                print(f"Resuming from sample {len(self.predictions)}...")
        else:
            self.error_count   = 0
            self.wrong_indexes = []
            self.predictions   = []

    def evaluate(self, verbose=False, save_every=10):
        """Run NL → ASP prediction over the test set and save checkpoints."""
        start_idx = len(self.predictions)

        for idx in tqdm(range(start_idx, len(self.test_dataset)), desc="Evaluating"):
            sample   = self.test_dataset[idx]
            input_nl = sample["messages"][1]["content"].replace(
                "Translate the following natural language to answer set programming: ", ""
            ).strip()

            predicted_asp = self.__predict(input_nl)

            if verbose:
                print(f"\n[{idx+1}/{len(self.test_dataset)}] NL: {input_nl}")
                print(f"  Predicted ASP: {predicted_asp}")

            self.predictions.append({
                "input_nl":      input_nl,
                "predicted_asp": predicted_asp
            })

            if (idx + 1) % save_every == 0 or (idx + 1) == len(self.test_dataset):
                self._save_progress()

        total_samples = len(self.test_dataset)
        if verbose:
            print(f"\nTotal samples : {total_samples}")
            print(f"Predictions saved to: {self.checkpoint_path}")

        return self.predictions

    def _save_progress(self):
        checkpoint = {
            "error_count":   self.error_count,
            "wrong_indexes": self.wrong_indexes,
            "predictions":   self.predictions
        }
        with open(self.checkpoint_path, 'w') as f:
            json.dump(checkpoint, f)

    def __predict(self, input_text: str, max_new_tokens=512) -> str:
        prompt = self.tokenizer.apply_chat_template(
            [
                {
                    "role": "system",
                    "content": "You are an expert in Translating the Natural language (NL) into Answer Set Programming (ASP) translation. Always provide precise, syntactically and semantically correct translations of NL into ASP."
                },
                {
                    "role": "user",
                    "content": f"Translate the following natural language to answer set programming: {input_text} "
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        return self.decoder.decode(prompt, max_new_tokens=max_new_tokens, temperature=0.1)


print("ASPTester defined.")

## 5 · Single NL Test

In [ ]:
single_NL = "Give Your Test NL Here."

# ── Normal (unconstrained) decoding ──────────────────────────────────────────
decoder: ASPDecoder = NaiveASPDecoder(model=model, tokenizer=tokenizer)

tester = ASPTester(
    decoder=decoder,
    tokenizer=tokenizer,
    test_dataset=[{
        "messages": [
            {
                "role": "system",
                "content": "You are an expert in Translating the Natural language (NL) into Answer Set Programming (ASP) translation. Always provide precise, syntactically and semantically correct translations of NL into ASP."
            },
            {
                "role": "user",
                "content": f"Translate the following natural language to answer set programming: {single_NL} "
            }
        ]
    }]
)

results = tester.evaluate(verbose=True)
print(f"\nPredictions: {tester.predictions}")

## 6 · ASPGenerator

In [ ]:

class ASPGenerator(ASPTester):
    def __init__(self, decoder: ASPDecoder, tokenizer: PreTrainedTokenizer, test_dataset):
        super().__init__(decoder, tokenizer, test_dataset)
        self.data_dict = []

    def process_and_save(
        self,
        json_path: str,
        output_filename: str = "path/to/your/output_file.csv",
        verbose: bool = True
    ):
        with open(json_path, 'r') as f:
            raw_data = json.load(f)

        samples = raw_data.get("data_dict", [])
        self.data_dict = []
        total_samples  = len(samples)

        print(f"Starting LLaMA-3 NL→ASP Pipeline: processing {total_samples} samples...")

        for item in tqdm(samples, disable=not verbose):
            nl_input   = item.get('NL_V2', '')
            actual_asp = item.get('ASP', '')
            category   = item.get('Category', 'N/A')
            item_id    = item.get('ID', item.get('Id', 'N/A'))

            # __predict is inherited from ASPTester (name-mangled)
            predicted_asp = self._ASPTester__predict(nl_input)

            self.data_dict.append({
                'Natural Language': nl_input,
                'Predicted ASP':    predicted_asp,
                'Actual ASP':       actual_asp,
                'Category':         category,
                'ID':               item_id
            })

        results_df = pd.DataFrame(self.data_dict)
        results_df.to_csv(output_filename, index=False)

        if verbose:
            print("\n" + "="*40)
            print("LLAMA-3 NL→ASP EVALUATION SUMMARY")
            print("="*40)
            print(f"Total Samples : {total_samples}")
            print(f"CSV saved to  : {output_filename}")
            print("="*40)

        return self.data_dict


print("ASPGenerator defined.")

## 7 · Run Full Evaluation Pipeline

In [ ]:
dataset_file = "path/to/your/test_dataset.json"

# ── Normal (unconstrained) decoding ──────────────────────────────────────────
decoder: ASPDecoder = NaiveASPDecoder(model=model, tokenizer=tokenizer)

generator = ASPGenerator(decoder=decoder, tokenizer=tokenizer, test_dataset=None)

results = generator.process_and_save(
    json_path=dataset_file,
    output_filename="path/to/your/output_file.csv"
)